# SC20 - Five-Fold Cross-Validation (ConvNeXt-S)

This workflow uses **K=5** and pools item-level predictions from five held-out test folds. The cohort size comes from the attached manifests. The supplied SC9 results belong to the historical 457-item cohort; a current dataset rerun must identify its own release and support. No completed SC20 outputs are included in this repository.

The notebook saves completed fold caches and supports resuming from its own attached output. A time guard limits additional training, and the final merge runs after all five fold caches are available. Session count and runtime depend on the GPU, recipe, and dataset.

## Run and resume

1. Attach `thesis-common-10c`, the raw image dataset, and the matching SC1 configuration, then select **Save Version (Save & Run All)**.
2. To resume, attach this notebook's previous output through **Add Input -> Your Work**. Use only fold caches from the same dataset, helper, and configuration.
3. Run again to train the remaining folds. Repeat until all five folds and merged artifacts are present.
4. When the log reports `still missing: NONE`, download the complete final output into `Results/SC20/`. Keep all per-fold caches alongside the merged results for coverage checks.


In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- configuration -----------------------------------------------------------------
# All 5 ConvNeXt-S folds together need ~16-18 h, i.e. MORE than one 12 h Kaggle session,
# and a 'Save & Run All' version that hits the 12 h wall FAILS and keeps NOTHING.
# This notebook is therefore SELF-RESUMING - you never edit anything between runs:
#   * fold files found in the inputs (from this notebook's own previous version) are
#     carried forward into the output,
#   * only the still-missing folds are trained, each saved IMMEDIATELY when it finishes,
#   * a time guard refuses to start a fold that might not finish before BUDGET_H,
#   * when all 5 fold files exist, the final merge runs automatically in the same session.
# The StratifiedKFold split is seeded, so folds are identical across sessions.
RUN_FOLDS = [0, 1, 2, 3, 4]   # normally leave as is - resume logic picks the missing ones
BUDGET_H  = 10.5              # stop starting new folds beyond this (batch cap is 12 h)
BACKBONE  = 'convnext_small.fb_in22k_ft_in1k'
K         = 5
VAL_FRAC  = 0.12
TAG       = 'convnext_small5'
print(f'SC20 5-fold CV | RUN_FOLDS={RUN_FOLDS} | budget {BUDGET_H}h | backbone={BACKBONE}')

In [ ]:
# ---- carry forward folds finished in previous sessions ------------------------------
# After the first run, add THIS notebook's own output as an input (+ Add Input -> Your
# Work -> this notebook). Every later run then starts from the folds already done.
import re, shutil
from collections import defaultdict

def fold_id(path):
    m = re.search(r'fold(\d+)_logits\.npz$', os.path.basename(path))
    return int(m.group(1)) if m else None

def _n_items(path):
    z = np.load(path, allow_pickle=True)
    return len(set(z['test_items'].tolist()))

# If more than one past SC20 output ends up attached at once (e.g. an old pre-dataset-
# -expansion version left attached alongside a newer one), don't silently take whichever
# sorts first alphabetically - pick the candidate with the MOST test items instead, since
# that's the one from the more complete/newer dataset.
_candidates = defaultdict(list)
for p in glob.glob('/kaggle/input/**/kfold5_fold*_logits.npz', recursive=True):
    fid = fold_id(p)
    if fid is not None:
        _candidates[fid].append(p)

for fid, paths in sorted(_candidates.items()):
    dst = os.path.join(OUT, f'kfold5_fold{fid}_logits.npz')
    if os.path.exists(dst):
        continue
    if len(paths) > 1:
        scored = sorted(((p, _n_items(p)) for p in paths), key=lambda x: -x[1])
        print(f'fold {fid}: {len(paths)} candidate files found (stale attachment from an '
              f'older dataset version?) - picking the one with the most test items:')
        for p, n in scored:
            print(f"    {'-> ' if (p, n) == scored[0] else '   '}{p}  ({n} items)")
        src = scored[0][0]
    else:
        src = paths[0]
    shutil.copy2(src, dst); print('carried forward:', os.path.basename(dst), '<-', src)

def folds_done():
    return sorted({fold_id(p) for p in glob.glob(f'{OUT}/kfold5_fold*_logits.npz')} - {None})
print('folds already available:', folds_done(), '| still missing:',
      sorted(set(range(K)) - set(folds_done())))

In [ ]:
# ---- train the still-missing folds; save each one IMMEDIATELY when it finishes ------
import time, torch, pandas as pd
from sklearn.model_selection import StratifiedKFold

todo = [f for f in RUN_FOLDS if f not in folds_done()]
if todo:
    ROOT = T.find_image_root()
    df_tr, df_va, df_te = T.load_manifests(ROOT)
    alldf = pd.concat([df_tr, df_va, df_te], ignore_index=True)
    orig  = alldf[alldf.source == 'original'].drop_duplicates('item_id')
    items = orig.item_id.to_numpy(); strat = orig.y_material.to_numpy()
    print('items entering CV:', len(items), '| folds to train this session:', todo)
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=T.SEED)
    HEADS = ['material', 'condition', 'route', 'threat']
    t0 = time.time(); fold_times = []
    for fold, (tr_idx, te_idx) in enumerate(skf.split(items, strat)):
        if fold not in todo:
            continue
        if fold_times:  # time guard: never start a fold that could blow the 12h cap
            eta = 1.15 * max(fold_times)
            if (time.time() - t0) + eta > BUDGET_H * 3600:
                print(f'TIME GUARD: {((time.time()-t0)/3600):.1f}h elapsed + ~{eta/3600:.1f}h/fold '
                      f'> {BUDGET_H}h budget -> NOT starting fold {fold}. '
                      f'Finished folds are saved; just run this notebook again.')
                break
        t_f = time.time()
        te_items = set(items[te_idx])
        rng = np.random.default_rng(T.SEED + fold)
        rem = rng.permutation(items[tr_idx])
        n_val = max(int(len(rem) * VAL_FRAC), K)
        va_items, trn_items = set(rem[:n_val]), set(rem[n_val:])
        tr_f = alldf[alldf.item_id.isin(trn_items)]
        va_f = alldf[(alldf.item_id.isin(va_items)) & (alldf.source == 'original')]
        te_f = alldf[(alldf.item_id.isin(te_items)) & (alldf.source == 'original')]
        T.check_leakage(tr_f, va_f, te_f)
        print(f'\n=== FOLD {fold+1}/{K} | train items {len(trn_items)} (rows {len(tr_f)}) '
              f'| val {len(va_items)} | test {len(te_items)} (rows {len(te_f)})')
        model, hist = T.train_backbone(BACKBONE, tr_f, va_f, te_f)
        _, _, te_loader = T.make_loaders(tr_f, va_f, te_f)
        o, y, it = T.evaluate_logits(model, te_loader)
        np.savez_compressed(f'{OUT}/kfold5_fold{fold}_logits.npz',
            **{f'test_log_{k}': o[k] for k in HEADS},
            **{f'test_y_{k}': y[k] for k in HEADS},
            test_items=np.array(it).astype(str), fold=np.array([fold]))
        fold_times.append(time.time() - t_f)
        print(f'fold {fold} saved -> kfold5_fold{fold}_logits.npz '
              f'({fold_times[-1]/3600:.2f} h; total {(time.time()-t0)/3600:.2f} h)')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('no folds to train in this session.')
missing = sorted(set(range(K)) - set(folds_done()))
print('\nfolds done:', folds_done(), '| still missing:', missing if missing else 'NONE - merging below')

In [ ]:
# ---- MERGE (automatic): when all 5 fold files exist, pool them into the final report ----
import torch, pandas as pd
if len(folds_done()) == K:
    HEADS = ['material', 'condition', 'route', 'threat']
    by_fold = {fold_id(p): p for p in glob.glob(f'{OUT}/kfold5_fold*_logits.npz')}
    print('merging folds:', {k: os.path.basename(v) for k, v in sorted(by_fold.items())})
    P = {k: [] for k in HEADS}; PY = {k: [] for k in HEADS}; PIT = []
    for fid in sorted(by_fold):
        z = np.load(by_fold[fid], allow_pickle=True)
        for k in HEADS:
            P[k].append(z[f'test_log_{k}']); PY[k].append(z[f'test_y_{k}'])
        PIT += list(z['test_items'])
    P  = {k: np.concatenate(v) for k, v in P.items()}
    PY = {k: np.concatenate(v) for k, v in PY.items()}
    PIT = np.array(PIT)
    uniq = set(PIT.tolist())
    # expected item count is derived from the CURRENTLY attached dataset (not hardcoded) so this
    # never goes stale again if the dataset is expanded/shrunk between CV runs
    _root = T.find_image_root()
    _tr, _va, _te = T.load_manifests(_root)
    _all = pd.concat([_tr, _va, _te], ignore_index=True)
    expected_n = _all[_all.source == 'original'].item_id.nunique()
    print('pooled rows:', len(PIT), '| unique items:', len(uniq), f'(expect {expected_n})')
    assert len(uniq) == expected_n, 'item coverage mismatch - duplicate or truncated fold files?'

    rep = T.full_report(P, PY, PIT)
    json.dump(rep, open(f'{OUT}/kfold_{TAG}_metrics.json', 'w'), indent=2)
    np.savez_compressed(f'{OUT}/kfold_{TAG}_logits.npz',
        **{f'test_log_{k}': P[k] for k in P},
        **{f'test_y_{k}': PY[k] for k in PY}, test_items=PIT.astype(str))
    s = rep['scalars']
    print('MATERIAL  acc=%.3f  macroF1=%.3f  ece=%.3f' % (s['material']['acc'], s['material']['macro_f1'], s['material']['ece']))
    print('ROUTE     macroF1=%.3f' % s['route']['macro_f1'])
    print('THREAT    QWK=%.3f' % s['threat']['qwk'])
    print('CONDITION AUROC=%.3f' % s['condition']['auroc'])

    # decision layer over every item (hazard recall) + bootstrap CI
    agg, uniq_i, idx = T.aggregate_by_item(P, PIT)
    lab   = T.item_labels(PY, PIT, idx)
    mprob = T.softmax_np(agg['material']); cprob = T.sigmoid_np(agg['condition'])
    tprob = T.corn_class_probs(torch.tensor(agg['threat'])).numpy()
    routes, _ = T.apply_policy(mprob, cprob, tprob)
    routes = np.asarray(routes, dtype=object); true_route = lab['route'].astype(int)
    base = T.decision_report(routes, true_route)
    rng = np.random.default_rng(T.SEED); n = len(uniq_i); ii = np.arange(n); hz, cst = [], []
    for _ in range(5000):
        sel = rng.choice(ii, size=n, replace=True)
        d = T.decision_report(routes[sel], true_route[sel])
        hz.append(d['hazard_recall']); cst.append(d['total_cost'] / d['n'])
    def _ci(a): a = np.array(a, float); return [round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4)]
    json.dump({'pooled': {k: (round(v, 4) if isinstance(v, float) else v) for k, v in base.items()},
               'hazard_recall_ci': _ci(hz), 'cost_per_item_ci': _ci(cst)},
              open(f'{OUT}/kfold_{TAG}_decision.json', 'w'), indent=2)
    print('hazard recall %.3f  CI %s' % (base['hazard_recall'], _ci(hz)))

    # bootstrap CIs of the headline metrics
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
    def _metrics(sel):
        mp = T.softmax_np(agg['material'][sel]).argmax(1)
        rp = T.softmax_np(agg['route'][sel]).argmax(1)
        cp = T.sigmoid_np(agg['condition'][sel])
        tp = T.corn_levels(torch.tensor(agg['threat'][sel])).numpy()
        ym, yr, yc, yt = lab['material'][sel], lab['route'][sel], lab['condition'][sel].astype(int), lab['threat'][sel]
        return {'material_acc': accuracy_score(ym, mp),
                'material_f1': f1_score(ym, mp, average='macro', labels=range(len(T.MATERIALS)), zero_division=0),
                'route_f1': f1_score(yr, rp, average='macro', labels=range(len(T.ROUTES)), zero_division=0),
                'threat_qwk': cohen_kappa_score(yt, tp, weights='quadratic', labels=list(range(T.NUM_THREAT))),
                'condition_auroc': roc_auc_score(yc, cp)}
    point = _metrics(ii); B = 5000; boot = {k: [] for k in point}
    for _ in range(B):
        sel = rng.choice(ii, size=n, replace=True)
        for k, v in _metrics(sel).items(): boot[k].append(v)
    ci = {}
    for k in point:
        a = np.array(boot[k], float); a = a[~np.isnan(a)]
        lo, hi = np.percentile(a, [2.5, 97.5])
        ci[k] = {'point': round(float(point[k]), 4), 'lo': round(float(lo), 4), 'hi': round(float(hi), 4)}
        print(f'{k:16s} {point[k]:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')
    json.dump({'n_items': n, 'n_bootstrap': B, 'ci': ci}, open(f'{OUT}/kfold_{TAG}_ci.json', 'w'), indent=2)
else:
    print(f'merge deferred: only folds {folds_done()} available. '
          'Save Version now, then run the notebook again (with its own output attached).')

## Expected outputs after all folds complete

`kfold5_fold0_logits.npz` through `kfold5_fold4_logits.npz`, plus:

- `kfold_convnext_small5_metrics.json`
- `kfold_convnext_small5_logits.npz`
- `kfold_convnext_small5_ci.json`
- `kfold_convnext_small5_decision.json`

Download the complete final output into `Results/SC20/`. Verify item coverage, fold consistency, dataset identity, and evaluation support before using these outputs in report tables or figures. An available notebook source does not establish a completed five-fold evaluation.
